<a href="https://colab.research.google.com/github/ss01-0-0/Pilot/blob/main/Python-Projects/Stock%20Market%20Data%20Presenter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries that i will use
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import logging as lg

# get user to input ticker symbol, start and end date
ticker_symbol=input("Ticker symbol to check: ")
start_date=input("Start date (YYYY-MM-DD): ")
end_date=input("End date (YYYY-MM-DD): ")

lg.getLogger("yfinance").setLevel(lg.CRITICAL)

# fetch data
data=yf.download(ticker_symbol,start=start_date,end=end_date)
if data.empty:
  print("""]
Error, data typed is in the wrong format or hasn't been found.
Please try again.""")
else:
  # compute analysis columns, and moving average signals (for using .describe()
  # later)
  data["Return"]=data["Close"].pct_change()
  data["MA20"]=data["Close"].rolling(20).mean()
  data["MA5"]=data["Close"].rolling(5).mean()
  data["Volatility"]=data["Return"].rolling(20).std()
  data["Signal"]=0
  data.loc[data["MA5"]>data["MA20"], "Signal"]=1
  data.loc[data["MA5"]<data["MA20"], "Signal"]=-1
  data["Signal Change"]=data["Signal"].diff()
  golden_cross=data[data["Signal Change"]==2]
  death_cross=data[data["Signal Change"]==-2]
  # backtest - strategy return versus buy and hold
  data["Strategy_Return"]= data["Signal"].shift(1) * data["Return"]
  cumulative_strategy=(1+data["Strategy_Return"]).cumprod()-1
  cumulative_holding=(1+data["Return"]).cumprod()-1
  final_strategy_return = cumulative_strategy.iloc[-1]
  final_holding_return = cumulative_holding.iloc[-1]
  print(f"\nStrategy return over period: {final_strategy_return:.2%}")
  print(f"Buy-and-hold return over period: {final_holding_return:.2%}")
  # plot the data (for close price, MA20, MA5) as well as golden cross and death
  # cross
  plt.figure(figsize=(10,5))
  plt.plot(data["Close"], label="Close Price", alpha=0.6)
  plt.plot(data["MA20"], label="20-day MA")
  plt.plot(data["MA5"], label="5-day MA")
  plt.scatter(golden_cross.index, golden_cross["Close"], color="green", marker="^", s=100, label="Golden Cross")
  plt.scatter(death_cross.index, death_cross["Close"], color="red", marker="v", s=100, label="Death Cross")
  plt.title(f"{ticker_symbol} Moving Crossover Signal")
  plt.xlabel("Date")
  plt.ylabel("Price")
  plt.legend()
  plt.show()
  # plot the data (for volatility)
  plt.figure(figsize=(10,3))
  plt.plot(data["Volatility"], label="20-day Volatility", color="orange")
  plt.title(f"{ticker_symbol} Rolling Volatility")
  plt.legend()
  plt.show()

# What I Learned
- Python becomes much more powerful through external libraries.
- yfinance provides an easy interface to retrieve historical stock market data.
- A pandas DataFrame stores tabular data.
- .head() returns the first five rows by default, making it useful for inspecting datasets quickly.
- .download() downloads the data from the given DataFrame.
- .describe() presents count, mean, quartiles etc.
- .rolling() when paired with a statistical function such as .mean() or .std() is a way of going through a subset while carrying out the statistical function paired with it.
- .figure is a more customisable way of creating an empty space to plot a graph on.
- .plot is how you can plot points or a line graph.
- learned what MA20, MA5, crossover signals, and close price mean for traders
- learned how to interpret volatility graphs and crossover signals in conjunction to decide on whether buying or selling is better
- learned how to use .loc[] and .scatter() with .diff() to plot crosses
- learned how to use logging as a Python library.
- learned how to use .shift() to move data in a dataframe.
- learned how to use .iloc() to locate integers in a dataframe.
- learned how to interpret results of backtesting to understand what the correct strategy in certain situations is.


---


# Version Changes
v.01:
- is able to import yfinance, pandas and matplotlib
- is able to access data from yfinance
- is able to display the first 5 rows from the dataset
- cannot use user input to pull specific data

v.02:
- matplotlib import is corrected to say matplotlib.pyplot
- allows user to enter ticker symbol, dates, and number of days to pull data for.
- is able to describe data.
- cannot present the data in graphs yet.
- columns are not aligned in the terminal (at least on my screen)

v.03:
- added the ability to plot data
- removed .describe() for now
- added variables for volatility, MA20 and returns
- for .std needed to add () so that it's not using the method itself, but is calling it
- code reformatted

v.04:
- added MA5, crossover signals (such as golden cross or death cross)
- added a separate graph showing volatility
- code reformatted
- if else check to test if the data inputted is correct (avoids crashes)
- to make it visually cleaner used logging to clear the screen of the error message should it come up

v.05:
- added backtesting (strategy return versus buy and hold)
- some minor typos corrected